# Microsoft Foundry · Agente pesquisador da web

Laboratório do curso de **Microsoft Foundry** (Seção 4 · Aula 14): criação, pelo código, de um agente com a ferramenta **Web Search**, capaz de responder com informações **atuais** da internet e indicar os **links** usados como fonte.

## Sumário

0. [Visão geral do projeto](#0.-Visão-geral-do-projeto)
1. [Instalação das bibliotecas](#1.-Instalação-das-bibliotecas)
2. [Variáveis de ambiente](#2.-Variáveis-de-ambiente)
3. [Conectando ao projeto do Foundry](#3.-Conectando-ao-projeto-do-Foundry)
4. [Cliente OpenAI do projeto](#4.-Cliente-OpenAI-do-projeto)
5. [Criando o agente com Web Search](#5.-Criando-o-agente-com-Web-Search)
6. [Criando a conversa](#6.-Criando-a-conversa)
7. [Perguntando ao agente](#7.-Perguntando-ao-agente)
8. [Verificando no portal do Foundry](#8.-Verificando-no-portal-do-Foundry)
9. [Resumo dos aprendizados](#9.-Resumo-dos-aprendizados)
10. [Limpeza e custos](#10.-Limpeza-e-custos)
11. [Interface com Streamlit](#11.-Interface-com-Streamlit)

---
## 0. Visão geral do projeto

### 0.1 Problema

Um modelo de linguagem só conhece o que viu no treinamento: perguntas sobre fatos recentes ("quais as novidades de 2025?") ficam desatualizadas ou inventadas, e não há como conferir de onde veio a informação.

### 0.2 Objetivo

- Criar um agente no **Foundry Agent Service** com a ferramenta **Web Search** (já disponível no Foundry, sem infraestrutura extra).
- Mostrar que o **próprio agente** analisa a intenção da pergunta e decide quando buscar na web.
- Receber respostas atuais **com citações** (URL e título de cada fonte).
- Conferir no **tracing** do portal a chamada da ferramenta e o consumo de tokens.

### 0.3 Ferramentas e serviços da Azure

| Serviço / ferramenta | Papel no projeto |
| --- | --- |
| **Microsoft Foundry** (projeto) | Reúne modelos, agentes, conversas e observabilidade. |
| **Foundry Agent Service** | Hospeda o agente `web-search-agent` e orquestra a chamada da ferramenta. |
| **Web Search tool** (`WebSearchPreviewTool`) | Ferramenta pronta do Foundry que busca na web e devolve trechos com URLs. Está em *preview*. |
| **Azure OpenAI** (deployment GPT-4) | Modelo que interpreta a pergunta e escreve a resposta. |
| **Microsoft Entra ID** | Autenticação sem chave no código (`DefaultAzureCredential`). |
| **Conversations + Tracing** | Histórico da conversa e rastreamento das chamadas no portal. |
| `azure-ai-projects`, `openai`, `python-dotenv`, `azure-identity`, `streamlit` | SDKs e interface. |

### 0.4 Como funciona (visão simples)

![Fluxo do agente com busca na web](fluxo_web_search.svg)

### 0.5 Arquitetura

![Arquitetura do projeto](arquitetura.svg)

> 💡 **Para um leigo:** o agente é um assistente que, ao perceber que a pergunta é sobre algo recente, "abre o navegador", lê algumas páginas e responde citando de onde tirou cada informação.

---
## 1. Instalação das bibliotecas

| Biblioteca | Para que serve |
| --- | --- |
| `azure-ai-projects==2.0.0b2` | **Foundry SDK v2**: cliente do projeto (`AIProjectClient`), agentes e ferramentas. Versão *beta*, fixada para o código funcionar igual ao da aula. |
| `openai==1.109.1` | Cliente OpenAI, usado para criar conversas e enviar mensagens ao agente (Responses API). |
| `python-dotenv` | Lê o arquivo `.env` com as configurações. |
| `azure-identity` | Autenticação na Azure sem chave no código (`DefaultAzureCredential`). |


> ⚠️ Versões fixadas (`==`) evitam surpresas: SDKs em *beta* mudam nomes de classes e métodos entre versões.

In [ ]:
# Instalando as bibliotecas no mesmo ambiente do kernel do notebook (%pip)
# azure-ai-projects = Foundry SDK | openai = conversa com o agente
# python-dotenv = lê o .env | azure-identity = login na Azure sem chave no código
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity

---
## 2. Variáveis de ambiente

As configurações ficam no arquivo **`.env`** (fora do código):

| Variável | O que é | Onde encontrar |
| --- | --- | --- |
| `FOUNDRY_PROJECT_ENDPOINT` | Endereço (URL) do projeto Foundry. | Portal do Foundry → página **Início** (*Overview*) → *Project endpoint*. |
| `MODEL_DEPLOYMENT_NAME` | Nome do deployment do modelo de chat. | Portal do Foundry → **Modelos + endpoints**. |


```
FOUNDRY_PROJECT_ENDPOINT=https://<seu-recurso>.services.ai.azure.com/api/projects/<seu-projeto>
MODEL_DEPLOYMENT_NAME=<nome-do-deployment>

```

> 🔒 **Segurança:** o `.env` guarda apenas **endereços e nomes** (nenhuma chave), porque a autenticação é feita pelo Microsoft Entra ID (seção seguinte). Se algum dia ele passar a ter segredos (chaves de API, tokens), não deve ser publicado: nesse caso, adicione `.env` ao `.gitignore`.

Neste notebook também importamos as classes do agente e da ferramenta:

| Classe | Para que serve |
| --- | --- |
| `PromptAgentDefinition` | Define o agente: modelo + instruções + ferramentas. |
| `WebSearchPreviewTool` | A ferramenta de busca na web. |
| `ApproximateLocation` | (opcional, não usada aqui) informa uma localização aproximada para resultados regionais. |

In [ ]:
# Importando as bibliotecas e lendo as configurações do arquivo .env
# os.getenv busca cada valor pelo nome; se a variável não existir, retorna None
# WebSearchPreviewTool = ferramenta de busca na web do Foundry
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, WebSearchPreviewTool, ApproximateLocation

load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

---
## 3. Conectando ao projeto do Foundry

O **`AIProjectClient`** é a porta de entrada do Python para o projeto no Foundry: por ele criamos agentes, listamos conexões e obtemos o cliente OpenAI.

- **`endpoint`**: o endereço do projeto (do `.env`).
- **`credential`**: o **`DefaultAzureCredential`** reaproveita um login que já existe na máquina (Azure CLI `az login`, conta Azure do VS Code, *Managed Identity* quando roda na Azure…). Assim, **nenhuma chave fica escrita no código**: quem autentica é o **Microsoft Entra ID**.

> 💡 Para rodar localmente, basta executar `az login` antes. O usuário precisa ter permissão no projeto (por exemplo, a função **Azure AI User**).

In [ ]:
# Criando o cliente do projeto Foundry: endpoint do .env + login da máquina (Entra ID)
# A conexão é "preguiçosa": erros de login só aparecem na primeira chamada real
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

---
## 4. Cliente OpenAI do projeto

O agente roda sobre um modelo **GPT** (Azure OpenAI). Para conversar com ele usamos a **Responses API** do SDK `openai`, e o próprio projeto entrega esse cliente já configurado com o endpoint e a autenticação do Foundry: `project_client.get_openai_client()`.

In [ ]:
# Obtendo o cliente OpenAI já autenticado no projeto Foundry
# É ele que cria conversas e envia mensagens ao agente (Responses API)
openai_client = project_client.get_openai_client()

---
## 5. Criando o agente com Web Search

O agente é criado com `agents.create_version` (cada execução gera uma nova versão). A novidade em relação a um agente simples é o parâmetro **`tools`**: uma **lista de ferramentas** que o agente pode usar.

| Campo | Valor | Significado |
| --- | --- | --- |
| `agent_name` | `web-search-agent` | Nome do agente no portal. |
| `model` | `MODEL_DEPLOYMENT_NAME` | O "cérebro" (GPT-4). |
| `instructions` | `"You are a helpful assistant that uses web search to answer user queries."` | *"Você é um assistente útil que usa a busca na web para responder às perguntas do usuário."* |
| `tools` | `[WebSearchPreviewTool()]` | Dá ao agente acesso à busca na web. |

> 💡 Ter a ferramenta **não obriga** o agente a usá-la: ele decide a cada pergunta. Citar a busca nas instruções apenas ajuda o modelo a lembrar que ela existe.

In [ ]:
# Criando o agente "web-search-agent" com a ferramenta de busca na web
# tools = lista de ferramentas que o agente PODE usar (ele decide quando)
agent = project_client.agents.create_version(
    agent_name="web-search-agent",
    definition=PromptAgentDefinition(
        model = model_deployment_name,
        instructions="You are a helpful assistant that uses web search to answer user queries.",
        tools = [
            WebSearchPreviewTool()
        ]
    )
)

---
## 6. Criando a conversa

A **conversation** é um objeto guardado na Azure que **armazena o histórico** entre o usuário e o agente, identificado pelo `conversation.id`. Todas as mensagens enviadas com o mesmo id fazem parte do mesmo diálogo, então o agente "lembra" o que já foi dito.

> 💡 No portal, o mesmo `conversation.id` aparece em **Traces**, com cada chamada de ferramenta e o consumo de tokens.

In [ ]:
# Criando a conversa que guarda o histórico (memória) entre usuário e agente
# create a conversation to use with the agent
conversation = openai_client.conversations.create()
print(f"Created conversation with id: {conversation.id}")

---
## 7. Perguntando ao agente

A pergunta pede informação recente. Em português: *"Quais são os avanços mais recentes em energia renovável?"*

In [ ]:
# Pergunta que exige informação atual (o modelo sozinho não teria)
user_query = "What are the latest advancements in renewable energy?"

Agora enviamos a pergunta ao agente com a **Responses API**:

| Parâmetro | Valor | Para que serve |
| --- | --- | --- |
| `conversation` | `conversation.id` | Grava a mensagem na conversa e envia o histórico ao modelo. |
| `extra_body` → `agent` | `{"name": "web-search-agent", "type": "agent_reference"}` | Diz **qual agente** responde. Sem versão informada, usa **a mais recente**. |
| `input` | pergunta do usuário | O texto enviado ao agente. |

A resposta final em texto fica em **`response.output_text`**. Repare que a pergunta **não manda** o agente usar a ferramenta: é o próprio agente que analisa a intenção e decide.

In [ ]:
# Enviando a pergunta ao agente: ele analisa a intenção, busca na web e resume
# response.output_text = texto final da resposta
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={
        "agent": {
            "name": "web-search-agent",
            "type": "agent_reference"
        }
    },
    input = user_query
)

print(f"Agent response: {response.output_text}")

### 7.1 O que aconteceu na demonstração da aula

O agente respondeu que o setor de energia renovável teve vários avanços em 2025 (painéis solares **bifaciais** de nova geração, fotovoltaica de **baixo custo**…) e incluiu os **links** dos sites usados. Abrindo os links, as informações conferem com o conteúdo das páginas.

✅ O agente buscou na web **sem que a pergunta pedisse isso**: ele analisou a intenção ("preciso de dados atuais") e escolheu a ferramenta sozinho.

---
## 8. Verificando no portal do Foundry

| Onde | O que aparece |
| --- | --- |
| **Agentes** → `web-search-agent` | Versão 1 e, em **Ferramentas**, a busca na web. No **YAML** do agente aparece `web_search_preview` na lista de `tools`. |
| **Traces** → `conversation.id` | A chamada da ferramenta (`web_search_call`), com a operação **search** e a consulta derivada da pergunta ("últimos avanços em energia renovável 2025"). |
| Mensagem de retorno | Uma lista de **anotações** do tipo `url_citation`, cada uma com **URL** e **título** (ex.: *"4 principais tendências em tecnologia de energia limpa em 2025"*). |
| Metadados | Cerca de **7.254 tokens** no total. |

> 💡 **Tokens:** o texto das páginas encontradas entra no contexto do modelo. Por isso uma pergunta curta pode consumir milhares de tokens: algo a monitorar em produção.

---
## 9. Resumo dos aprendizados

- Um agente do Foundry recebe ferramentas pelo parâmetro **`tools`** do `PromptAgentDefinition`.
- A **Web Search** é uma ferramenta pronta do Foundry: basta `WebSearchPreviewTool()`, sem conexão ou chave extra.
- O agente **analisa a intenção** e decide sozinho quando chamar a ferramenta.
- As fontes voltam como anotações **`url_citation`** (URL + título): respostas atuais e **verificáveis**.
- O **tracing** mostra a consulta feita na web e o custo em tokens, que cresce com o conteúdo das páginas.

---
## 10. Limpeza e custos

| Recurso | Gera custo? |
| --- | --- |
| Chamadas ao modelo (tokens) | Sim, **por uso**: só quando você conversa com o agente. Ferramentas que trazem muito texto (páginas, APIs, documentos) aumentam bastante os tokens. |
| Ferramenta Web Search | Sim, **por uso**: as buscas têm cobrança própria além dos tokens do modelo (confira a página de preços do Foundry). |

Ao terminar os estudos:

- **Portal do Foundry → Agentes**: apagar as versões do agente que não serão mais usadas.
- Ao final do curso, apagar o **grupo de recursos** no `portal.azure.com` remove tudo de uma vez.

> 💡 Vale criar um **orçamento com alerta** em *Cost Management → Budgets* para ser avisado de qualquer gasto inesperado.

---
## 11. Interface com Streamlit

A célula abaixo gera o arquivo **`app.py`**: o mesmo agente, agora com uma interface de chat no navegador. Ela reaproveita o código do notebook:

| Função do `app.py` | Código do notebook reaproveitado |
| --- | --- |
| `criar_clientes()` | `AIProjectClient` + `DefaultAzureCredential` (seção 3) e `get_openai_client()` (seção 4) |
| `nova_conversa()` | `conversations.create()` (seção 6) |
| `perguntar()` | `responses.create` com `agent_reference` (seção 7) |
| `extrair_fontes()` | lê as anotações `url_citation` vistas no tracing (seção 8) |

Pontos importantes:

- O Streamlit **reexecuta o script inteiro** a cada interação. Por isso os clientes ficam em cache (`@st.cache_resource`) e o `conversation.id` e o histórico ficam no `st.session_state`.
- O agente **não é recriado** a cada mensagem: o app usa o agente já criado neste notebook (pelo nome, sempre na versão mais recente).
- As **fontes** (links) de cada resposta aparecem em um bloco expansível logo abaixo da mensagem.

**Como executar:**

```bash
pip install streamlit
az login
streamlit run app.py
```

O app abre em `http://localhost:8501` e usa o mesmo arquivo `.env` desta pasta.

In [ ]:
%%writefile app.py

import os

import streamlit as st
from azure.ai.projects import AIProjectClient
from azure.core.exceptions import ClientAuthenticationError, HttpResponseError
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from openai import APIError

# ------------------------------------------------------------------
# 1. Configurações (as mesmas do notebook)
# ------------------------------------------------------------------
load_dotenv()  # lê o .env da mesma pasta

FOUNDRY_PROJECT_ENDPOINT = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT_NAME = os.getenv("MODEL_DEPLOYMENT_NAME")
AGENT_NAME = "web-search-agent"  # agente criado na seção 5 do notebook

st.set_page_config(page_title="Web Search Agent · Microsoft Foundry", page_icon="🌐", layout="wide")


# ------------------------------------------------------------------
# 2. Conexão com o projeto do Foundry
# ------------------------------------------------------------------
# cache_resource: os clientes são criados uma única vez e reaproveitados
@st.cache_resource
def criar_clientes():
    project_client = AIProjectClient(
        endpoint=FOUNDRY_PROJECT_ENDPOINT,
        credential=DefaultAzureCredential(),
    )
    return project_client.get_openai_client()


# ------------------------------------------------------------------
# 3. Conversa e pergunta ao agente
# ------------------------------------------------------------------
def nova_conversa():
    # Nova conversation = histórico zerado; o id fica no session_state
    st.session_state.conversation_id = criar_clientes().conversations.create().id
    st.session_state.messages = []


def perguntar(texto):
    # Mesmo responses.create do notebook: o agente decide sozinho se busca na web
    return criar_clientes().responses.create(
        conversation=st.session_state.conversation_id,
        extra_body={"agent": {"name": AGENT_NAME, "type": "agent_reference"}},
        input=texto,
    )


# ------------------------------------------------------------------
# 4. Fontes: as URLs que a busca na web usou (anotações url_citation)
# ------------------------------------------------------------------
def extrair_fontes(response):
    fontes = {}
    for item in response.output:
        if item.type != "message":
            continue
        for conteudo in item.content:
            for anotacao in getattr(conteudo, "annotations", None) or []:
                if anotacao.type == "url_citation":
                    fontes[anotacao.url] = anotacao.title or anotacao.url  # dicionário evita links repetidos
    return [{"url": url, "titulo": titulo} for url, titulo in fontes.items()]


def mostrar_fontes(fontes):
    if fontes:
        with st.expander(f"🔗 Fontes da web ({len(fontes)})"):
            for fonte in fontes:
                st.markdown(f"- [{fonte['titulo']}]({fonte['url']})")


# ------------------------------------------------------------------
# 5. Interface
# ------------------------------------------------------------------
st.title("🌐 Agente pesquisador da web")
st.caption("Agente do Foundry Agent Service com a ferramenta Web Search: respostas atuais, com links das fontes.")

if not FOUNDRY_PROJECT_ENDPOINT or not MODEL_DEPLOYMENT_NAME:
    st.error("Configure FOUNDRY_PROJECT_ENDPOINT e MODEL_DEPLOYMENT_NAME no arquivo `.env`.")
    st.stop()

if "conversation_id" not in st.session_state:
    try:
        nova_conversa()
    except ClientAuthenticationError:
        st.error("Falha no login da Azure. Execute `az login` no terminal e recarregue a página.")
        st.stop()
    except (HttpResponseError, APIError) as erro:
        st.error(f"Erro ao criar a conversa no Foundry: {erro}")
        st.stop()

with st.sidebar:
    st.header("⚙️ Agente")
    st.caption(f"Agente: `{AGENT_NAME}`  \nModelo: `{MODEL_DEPLOYMENT_NAME}`  \nConversa: `{st.session_state.conversation_id}`")
    if st.button("🗑️ Nova conversa"):
        nova_conversa()
        st.rerun()
    st.divider()
    st.markdown("**Sugestões**")
    st.caption("• Quais são os avanços mais recentes em energia renovável?  \n• Quais as novidades do Microsoft Foundry este mês?")

# Reexibe o histórico
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])
        mostrar_fontes(message.get("fontes"))

if prompt := st.chat_input("Pergunte algo que precise de informação atual"):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        with st.spinner("Pesquisando na web..."):
            try:
                response = perguntar(prompt)
            except ClientAuthenticationError:
                st.error("Falha no login da Azure. Execute `az login` no terminal.")
            except (HttpResponseError, APIError) as erro:
                st.error(f"Erro ao consultar o agente: {erro}")
            else:
                fontes = extrair_fontes(response)
                st.markdown(response.output_text)
                mostrar_fontes(fontes)
                st.session_state.messages.append({"role": "assistant", "content": response.output_text, "fontes": fontes})
